파인튜닝 코드

In [1]:
!pip install pydub
!pip install imageio-ffmpeg

In [2]:
!pip install python-dotenv

In [ ]:
import os
import glob
from google.colab import drive
from datasets import Dataset, Audio, load_from_disk
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# 드라이브 마운트
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

raw_dataset_path = '/content/drive/MyDrive/AI_Project/whisper_raw_dataset'
drive_zip_path = '/content/drive/MyDrive/AI_Project/dataset_processed.zip'
local_dataset_path = '/content/dataset_processed'

if os.path.exists(raw_dataset_path):
    print("드라이브에 이미 저장된 Raw Dataset이 있습니다! 불러오는 중입니다...")
    dataset = load_from_disk(raw_dataset_path)
else:
    print("Raw Dataset이 없습니다. 데이터를 조립합니다.")

    local_wav_count = len(glob.glob(f"{local_dataset_path}/*.wav"))

    if local_wav_count < 64235:
        print("구글 드라이브의 ZIP 파일을 코랩 SSD에 직접 풉니다")
        print("(약 2~5분 정도 소요됩니다)")

        os.system(f"unzip -q -o '{drive_zip_path}' -d '/content/'")
        print("초고속 압축 해제 완료!")
    else:
        print(f"이미 로컬 SSD에 압축이 풀려있습니다 ({local_wav_count}개).")

    # 구글 드라이브가 아닌 '로컬 SSD' 경로에서 파일 찾기
    wav_files = sorted(glob.glob(f"{local_dataset_path}/*.wav"))
    txt_files = sorted(glob.glob(f"{local_dataset_path}/*.txt"))
    print(f"발견된 오디오: {len(wav_files)}개 / 정답지: {len(txt_files)}개")

    if len(wav_files) == 0 or len(txt_files) == 0:
        raise ValueError("로컬 SSD에 파일이 없습니다. 드라이브에 dataset_processed.zip 파일이 복원되었는지 확인해주세요!")

    def read_text(path):
        with open(path, 'r', encoding='utf-8') as f:
            return f.read().strip()

    print(f"{len(txt_files)}개 파일에 대해 읽기를 시작합니다...\n")

    with ThreadPoolExecutor(max_workers=32) as executor:
        sentences = list(tqdm(
            executor.map(read_text, txt_files),
            total=len(txt_files),
            desc="텍스트 읽기 진행률",
            unit="개"
        ))

    print("\n텍스트와 오디오를 Dataset에 조립합니다...")
    dataset = Dataset.from_dict({"audio": wav_files, "sentence": sentences})
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

    # 완성품을 다시 구글 드라이브로 피드백 저장
    print(f"조립된 완제품을 구글 드라이브 창고에 영구 저장합니다...")
    dataset.save_to_disk(raw_dataset_path)
    print("작업이 끝났습니다")

In [ ]:
!pip install evaluate jiwer

In [ ]:
import os
from google.colab import drive
from datasets import load_from_disk
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

raw_dataset_path = '/content/drive/MyDrive/AI_Project/whisper_raw_dataset'
print("구글 드라이브에서 rawdatset 패키지를 불러옵니다...")
dataset = load_from_disk(raw_dataset_path)

# 깨진 파일 먼저 걸러냅니다
def is_valid_audio(example):
    try:
        return len(example["audio"]["array"]) > 0
    except Exception:
        return False

num_cores = os.cpu_count() or 2
print("불량 오디오 파일 사전 검열을 시작합니다... (깨진 파일 솎아내기)")

clean_dataset = dataset.filter(is_valid_audio, num_proc=num_cores)
print(f"검열 완료! 총 {len(dataset)}개 중 살아남은 우량 데이터: {len(clean_dataset)}개\n")

# Whisper 전처리 도구 세팅
model_name_or_path = "openai/whisper-base"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_or_path)
tokenizer = WhisperTokenizer.from_pretrained(model_name_or_path, language="Korean", task="transcribe")

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

print("전처리를 시작합니다")

ready_dataset = clean_dataset.map(
    prepare_dataset,
    remove_columns=clean_dataset.column_names,
    num_proc=num_cores
)

# 완성품 영구 저장
processed_save_path = '/content/drive/MyDrive/AI_Project/whisper_ready_dataset'
print(f"전처리 완료된 최종 데이터를 구글 드라이브({processed_save_path})에 영구 저장합니다...")
ready_dataset.save_to_disk(processed_save_path)
print("영구 저장을 완료했습니다.")

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
import numpy as np
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained("openai/whisper-base", language="Korean", task="transcribe")

# 오디오 데이터를 묶기
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 성능 측정 세팅 (WER: 단어 에러율 - 낮을수록 좋은 모델)
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("오디오 포장 & 성능 측정 세팅 완료")

In [ ]:
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

print("Whisper AI Model 다운로드 중...")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

eval_subset = ready_dataset.select(range(500))
print(f"1000번마다 진행될 모의고사 데이터 개수: {len(eval_subset)}개 세팅 완료!")

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/AI_Project/whisper_finetuned",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=1,
    fp16=True,
    gradient_checkpointing=True,

    eval_strategy="steps",
    predict_with_generate=True,
    generation_max_length=225,

    logging_steps=100,
    eval_steps=1000,
    save_steps=1000,
    save_total_limit=2,

    # 로그인 입력창 방지
    report_to="none"
)

# 훈련 시작
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=ready_dataset,
    eval_dataset=eval_subset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

print("Fine-tuning을 시작합니다...")
trainer.train()

In [ ]:
# ===================================================================================
# 테스트 코드
# ===================================================================================

In [ ]:
import torch
from transformers import pipeline

print("AI 실행중...")
# [drive] > [MyDrive] > [AI_Project] > [whisper_finetuned] 안에 있는
# 가장 마지막 번호의 checkpoint 폴더 이름을 확인하고 아래 경로 숫자 지정
model_path = "/content/drive/MyDrive/AI_Project/whisper_finetuned/checkpoint-8022"

# 파이프라인 생성
stt_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model_path,
    tokenizer="openai/whisper-base",
    feature_extractor="openai/whisper-base",
    device=0 if torch.cuda.is_available() else -1
)

# 테스트할 오디오 파일 경로 지정
audio_file = "/content/test.wav"

print(f"AI가 [{audio_file}] 파일을 듣고 있습니다. 잠시만 기다려주세요...")

# STT 시작
result = stt_pipeline(audio_file, generate_kwargs={"language": "korean"})

print("\n==================================")
print("AI 받아쓰기 최종 결과")
print("====================================")
print(result["text"])
print("====================================\n")